# Demo on Tuning GCN Surrogate Model

###  Introduction
This notebook demonstrates how to tube a Graph Convolutional Network (GCN) for performance of parametrized quantum circuits (PQCs).


### Imports and Setup

First, we suppress PyTorch Warnings and ensure proper dependency loading. Then, we import core modules and helper functions from our project structure.

In [1]:
# Suppress PyTorch Warnings and ensure proper dependency loading
import warnings
import os
import sys

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning)
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..')) # Go two levels up from the notebook location to reach project root

if project_root not in sys.path:
    sys.path.insert(0, project_root)
print("Project root set to:", project_root)

Project root set to: /Users/danielbarta/Desktop/work/FOKUS/SQuASH


In [2]:
import os
import torch
import optuna
import json

import torch.nn as nn

from functools import partial

from util.split import train_val_test_split
from util.data_loader import load_data
from surrogate_models.architectures.gnn.gcn_runner import prepare_paths_and_config, prepare_dataloaders, set_seed, save_json
from surrogate_models.tuning.gcn_tuning import objective


### 📁 Step 1: Prepare Paths and Configuration

The configuration for the surrogate model is defined and loaded in two stages:
 - setting the search space and device
 - specify config for the surrogate model

In [3]:
search_space = 'ghz_a'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

This sets the quantum circuit benchmark domain (search_space) and selects the device (cpu or cuda) for training.


In [4]:
config, gate_set, timestamp = prepare_paths_and_config(search_space, device)
print("Config (incl. model config):")
print(json.dumps(config, indent=4, default=str))

config["epochs"] = 3 # set the max.number of training epochs to 4

Config (incl. model config):
{
    "device": "cpu",
    "seed": 42,
    "runseed": 42,
    "batch_size": 32,
    "num_workers": 0,
    "epochs": 3,
    "emb_dim": 1200,
    "layer_num": 8,
    "qubit_num": 3,
    "num_node_features": 7,
    "drop_ratio": 0.012714767230404513,
    "lr": 0.00042048670814195114,
    "decay": 1.2239395743425164e-06,
    "JK": "mean",
    "patience": 7,
    "metric": "spearman",
    "graph_pooling": "max",
    "n_estimators": null,
    "max_depth": null,
    "random_state": null,
    "optuna_trials": null,
    "min_samples_split": null,
    "min_samples_leaf": null,
    "max_features": null,
    "n_jobs": null,
    "PATHS": {
        "optuna_studies": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/surrogate_models/tuning/studies",
        "raw_data": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/raw_data/",
        "gcn_data": "/Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/processed_data/gcn_processed_data",
        "rf_data": "/Users/danielbarta/De

### 🔧 Step 2: Set Random Seed

For reproducibility, it's important to set all random seeds.

In [5]:
set_seed(config["runseed"])

### 📦 Step 3: Load your Dataset

To be processed by GCN, the data—i.e. quantum circuits—need to be transformed into DAGs. If preprocessed data is not found, it's automatically created from raw quantum circuit files.

In [6]:
data_name = f"gcn_demo_dataset_ghz_a"
data_path = os.path.join(config['PATHS']['gcn_data'], f'{data_name}.pt')
data = load_data(data_path)
train_data, val_data, test_data = train_val_test_split(data, train_size=0.8, val_size=0.15, shuffle=True, random_seed=None)


Data successfully loaded from /Users/danielbarta/Desktop/work/FOKUS/SQuASH/data/processed_data/gcn_processed_data/gcn_demo_dataset_ghz_a.pt


### 🧪 Step 4: Prepare DataLoaders

We use PyTorch Geometric's `DataLoader` to batch and feed our graph data efficiently into the GCN.

In [7]:
train_loader, val_loader, test_loader = prepare_dataloaders(
    train_data, val_data, test_data,
    batch_size=config['batch_size'],
    num_workers=config['num_workers']
)
# train_loader = val_loader + test_loader

### 📝 Step 5: Set up Tuner

We persist the configuration used for this training session for reproducibility.


In [8]:
model_name = f"gcn_demo_{search_space}_{timestamp}"
model_logs_path = os.path.join(config['PATHS'][f'trained_models'], f'{model_name}')
os.makedirs(model_logs_path, exist_ok=True)
save_json(os.path.join(model_logs_path, f'{model_name}_config.json'), config)

We set up MSE loss for training and a MedianPruner (with ten warmup steps) so Optuna can stop underperforming trials early. We then use `functools.partial` to bind configuration, data loaders, and the loss function into the objective so Optuna only needs to pass the trial argument, and specifies `direction = 'maximize'` because we’re optimizing Spearman’s `$\rho$`. Finally, we constructs a unique study name from the model name and points `storage` to an `SQLite` file so that all trial results are saved, resumable and can resumable, and can be utilized for parallelization..

In [9]:
# Define the loss function for regression
criterion = nn.MSELoss()

# Set up the Optuna pruner using median pruner strategy
pruner = optuna.pruners.MedianPruner(n_warmup_steps=10)

# Bind additional arguments to the objective function using partial.
objective_with_args = partial(objective, config=config, train_loader=train_loader, val_loader=val_loader, criterion=criterion)

# Define study direction based on the optimization metric
direction = 'maximize' # we want to increase spearman's rho
study_name = f"study_{model_name}"
storage = f"sqlite:///{os.path.join(config['PATHS']['optuna_studies'], study_name)}.db"


### 🚀 Step 6: Tune the GCN Model
We tuna a `RegGNN` model for a fixed number of trials.

In [10]:
# define number of trials
n_trials = 2

# Create or load an existing Optuna study
study = optuna.create_study(
    direction=direction,
    study_name=study_name,
    storage=storage,
    load_if_exists=True,
    pruner=pruner,
)

# Run optimization across a specified number of trials
study.optimize(objective_with_args, n_trials=n_trials)

print("Best hyperparameters: ", study.best_params)


[I 2025-06-04 11:25:18,529] A new study created in RDB with name: study_gcn_demo_ghz_a_2025-06-04_11-25-16


Tuning parameters: {'emb_dim': 1300, 'layer_num': 8, 'drop_ratio': 0.3903784769559986, 'graph_pooling': 'sum', 'lr': 6.329488522397883e-06, 'decay': 0.0023894413009024466, 'JK': 'last'}

Epoch 1/3


Validation: 100%|██████████| 87/87 [00:02<00:00, 34.64it/s]


Train loss: 1.0227, Val loss: 0.0492
Train Spearman: -0.0075, Val Spearman: -0.0819

Epoch 2/3


Validation: 100%|██████████| 87/87 [00:02<00:00, 33.43it/s]


...

Train loss: 0.3576, Val loss: 0.0596
Train Spearman: -0.0134, Val Spearman: 0.0037

Epoch 2/3


Validation: 100%|██████████| 87/87 [00:01<00:00, 52.61it/s]


Train loss: 0.1737, Val loss: 0.0546
Train Spearman: -0.0065, Val Spearman: 0.0342

Epoch 3/3


Validation: 100%|██████████| 87/87 [00:01<00:00, 64.17it/s]
[I 2025-06-04 11:28:56,403] Trial 1 finished with value: 0.05212192022542101 and parameters: {'emb_dim': 1350, 'num_layers': 4, 'drop_ratio': 0.4959754248048782, 'graph_pooling': 'max', 'lr': 1.4663498369851827e-06, 'decay': 0.016517879648429365, 'JK': 'mean'}. Best is trial 0 with value: 0.05983825297031728.


Train loss: 0.1621, Val loss: 0.0526
Train Spearman: 0.0018, Val Spearman: 0.0521
Best hyperparameters:  {'emb_dim': 1300, 'num_layers': 8, 'drop_ratio': 0.3903784769559986, 'graph_pooling': 'sum', 'lr': 6.329488522397883e-06, 'decay': 0.0023894413009024466, 'JK': 'last'}
